# Comparing unsupervised methods statistically

**Accompanies Section 9 of** *Best Practices for Unsupervised Learning in Molecular Systems* (Article v1.0).

A claim that one pipeline is better than another is a statistical claim. This notebook runs the full protocol end to end on real molecular data: five-fold cross-validation grouped by molecular formula, paired tests of every pipeline against a reference declared in advance, correction for multiple comparisons, effect sizes with confidence intervals, and an explicit practical-significance threshold.

### Learning objectives
- Generate one set of folds shared across every method, grouped so that constitutional isomers cannot straddle a split, and check that the folds are what you think they are
- Score unsupervised pipelines without labels
- Test every pipeline against a pre-declared reference with a paired t-test on the per-fold differences, then apply the Holm correction
- Say when repeated cross-validation needs a corrected variance, and why a grouped split with few groups cannot be repeated at all
- Distinguish statistical from practical significance
- See how easily an unpaired, uncorrected comparison invents a winner

### What this notebook is designed to make go wrong
A protocol that runs correctly, reports faithfully, and crowns the pipeline the rest of the guide tells you to trust least.

### What you need installed
numpy, scipy, scikit-learn, matplotlib, and ASE to read the QM7 structure file.

### Roughly how long it takes
About five minutes. Section 2 fits five pipelines on five held-out folds, t-SNE among them, and prints a line as each fold finishes.

In [ ]:
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats as sps

# One palette for every figure here. The series stay distinguishable in
# grayscale as well as in color, and marker and dash vary alongside the color,
# so nothing depends on color alone.
TEAL, PURPLE, LAVENDER, GREEN, PLUM, SLATE = (
    "#2D4F54", "#7B539E", "#B8A0D2", "#5A9448", "#9E4A78", "#3A3D4A"
)
PALETTE = [TEAL, PURPLE, LAVENDER, GREEN]


def set_style():
    """Apply the plot style used throughout these notebooks."""
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "axes.edgecolor": SLATE,
        "axes.labelcolor": SLATE,
        "axes.titlecolor": SLATE,
        "axes.linewidth": 1.0,
        "axes.grid": False,
        "xtick.color": SLATE,
        "ytick.color": SLATE,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "legend.frameon": False,
        "axes.prop_cycle": (
            mpl.cycler(color=PALETTE)
            + mpl.cycler(marker=["o", "s", "^", "D"])
            + mpl.cycler(linestyle=["-", (0, (4, 1.5)), (0, (1, 1.2)), (0, (5, 1.2, 1, 1.2))])
        ),
    })


set_style()
warnings.filterwarnings("ignore", category=FutureWarning)

# Fix a seed so the notebook reproduces. That is not the same as checking a
# conclusion survives a different seed, which we do explicitly where it matters.
SEED = 20260726
rng = np.random.default_rng(SEED)

# The example data, fetched by scripts/download_data.py.
DATA = Path.cwd().parent / "data"

## 1. What are we comparing, and on what metric?

Declare this **before** looking at any results. Choosing the metric after seeing the numbers is
itself a selection procedure.

The task is a clustering, not a prediction. Every arm partitions the *same* molecules, the 1219
fifteen-atom QM7 structures used in Sections 6, 7 and 8 (represented by their 105 off-diagonal
Coulomb matrix elements), into k-means groups, and the arms differ only in what is done to the
features first. There is no property being predicted here and no accuracy to report, because
there is no label to score against.

What replaces accuracy is **cluster stability**: cluster the data, draw a bootstrap resample,
cluster that, and measure how much of each original cluster reappears (its Jaccard overlap). A
pipeline scores high when the groups it finds come back in a fresh sample and low when they are
an artifact of the particular sample, which is the label-free stand-in for the per-fold test
accuracy a supervised comparison would use. Stability is reproducibility, not correctness: a
stable clustering can still be chemically empty, a point the results below make concrete.

Two of the arms differ only in the number of principal components retained. "How many
components?" is a decision people make by habit, and a comparison protocol should be able to
tell you whether the decision matters.

### Load the molecules the comparison runs on

`data/qm7.xyz` is an extended XYZ file: one frame per molecule, with the atomization energy on
the comment line. ASE reads the whole file in one call, so nothing here has to parse a file
format by hand.

Those coordinates are in Bohr, not angstrom. Everything below uses off-diagonal Coulomb matrix
entries, and scaling every distance by the same constant scales all of those entries by that
constant, so no result in this notebook depends on the unit. It would matter the moment you set
a cutoff in angstrom.

In [ ]:
from ase.io import read

# The same molecules as Sections 6-8 of the article. Restricting to a fixed
# atom count avoids the zero-padding artifact of Section 6.
frames = read(str(DATA / "qm7.xyz"), index=":")
fifteen = [atoms for atoms in frames if len(atoms) == 15]
positions = np.array([atoms.get_positions() for atoms in fifteen])
numbers = np.array([atoms.get_atomic_numbers() for atoms in fifteen], dtype=float)
iu = np.triu_indices(15, 1)


def coulomb_offdiag(pos, num):
    """The off-diagonal Coulomb matrix entries Z_i Z_j / r_ij, as one vector."""
    d = np.linalg.norm(pos[:, None] - pos[None, :], axis=-1)
    np.fill_diagonal(d, np.inf)
    return (np.outer(num, num) / d)[iu]


X = np.array([coulomb_offdiag(p, n) for p, n in zip(positions, numbers)])

# One independent observation here is a molecular FORMULA, not a molecule:
# constitutional isomers share a composition and resemble one another far more
# than they resemble the rest of the set. Grouping by formula keeps all isomers
# of one formula on the same side of every split (Section 5).
formula = np.array(["-".join(str(int(z)) for z in sorted(n)) for n in numbers])

print(f"data: {X.shape[0]} molecules x {X.shape[1]} features, "
      f"{len(set(formula))} distinct formulae")
print(f"feature standard deviations span {X.std(axis=0).min():.3f} to "
      f"{X.std(axis=0).max():.2f}, a factor of {X.std(axis=0).max() / X.std(axis=0).min():.0f}")

### Score each resample with bootstrap cluster stability

There are no labels here, so the per-resample score has to be label free. Cluster stability asks
the question a scientist means by "are these clusters real": would they come back in a fresh
sample? Cluster the data, resample it with replacement, cluster again, match the new clusters to
the original ones, and record how much of each original cluster survives. The whole pipeline is
rebuilt on each resample, not only k-means, so a step with no out-of-sample transform is refit
from scratch every time and pays for it in the score.

Two details in the function below are easy to get wrong, and both push the number upward. Match
clusters by optimal one-to-one assignment, not greedily, or two original clusters can be scored
against the same bootstrap cluster. And cluster the resampled multiset with its repeated draws,
then score only on the samples actually drawn; clustering the deduplicated draw turns a bootstrap
into a 63% subsample and lifts stability by a few hundredths, right where the reading thresholds
sit.

Those thresholds, by convention: above 0.85 highly stable, 0.75 to 0.85 stable, 0.60 to 0.75
weak, below 0.60 the cluster dissolved and should not be interpreted as a group. Those bands
are conventions and not tests, and a stable cluster can still be an artifact of a systematic
curation error, because stability measures reproducibility and not correctness.

In [ ]:
from scipy.optimize import linear_sum_assignment


def jaccard(a, b):
    """Jaccard overlap between two sets of sample indices."""
    a, b = np.unique(a), np.unique(b)
    intersection = np.intersect1d(a, b, assume_unique=True).size
    union = a.size + b.size - intersection
    return float(intersection / union) if union else 0.0


def cluster_stability(X, cluster_fn, n_bootstrap=100, random_state=0):
    """Size-weighted mean bootstrap Jaccard stability of a clustering.

    `cluster_fn` maps a data matrix to integer labels and has to accept data
    sets of different sizes. Fixing its seed, as we do below, isolates the
    variation being measured to resampling instead of initialization.

    The overall number is weighted by cluster size, so one tiny unstable
    cluster cannot dominate it.
    """
    X = np.asarray(X, dtype=float)
    rng = np.random.default_rng(random_state)

    reference = np.asarray(cluster_fn(X))
    reference_sets = [np.where(reference == lab)[0] for lab in np.unique(reference[reference >= 0])]
    sizes = np.array([s.size for s in reference_sets])
    scores = np.zeros((n_bootstrap, len(reference_sets)))

    for b in range(n_bootstrap):
        idx = rng.choice(X.shape[0], size=X.shape[0], replace=True)
        present = np.unique(idx)
        # Cluster the full resampled multiset, repeated draws included: those
        # repeats carry the weight that makes this a bootstrap. Score only on
        # the samples actually drawn, since one that was never drawn cannot be
        # recovered.
        boot = np.asarray(cluster_fn(X[idx]))
        boot_sets = [np.unique(idx[boot == lab]) for lab in np.unique(boot[boot >= 0])]

        # Optimal one-to-one matching on the Jaccard overlap matrix. Greedy
        # best-match can send two original clusters to the same bootstrap
        # cluster, which inflates stability.
        cost = np.zeros((len(reference_sets), len(boot_sets)))
        for i, ref in enumerate(reference_sets):
            ref_present = np.intersect1d(ref, present, assume_unique=True)
            for j, bs in enumerate(boot_sets):
                cost[i, j] = -jaccard(ref_present, bs)
        rows, cols = linear_sum_assignment(cost)
        for i, j in zip(rows, cols):
            scores[b, i] = -cost[i, j]

    per_cluster = scores.mean(axis=0)
    return float(np.sum(per_cluster * sizes / sizes.sum()))

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler


def embed(data, kind):
    """Everything except the unscaled arm is standardized first."""
    if kind == "unscaled + k-means":
        return data
    scaled = StandardScaler().fit_transform(data)
    if kind == "standardized + k-means":
        return scaled
    if kind == "PCA(2) + k-means":
        return PCA(n_components=2, random_state=SEED).fit_transform(scaled)
    if kind == "PCA(3) + k-means":
        return PCA(n_components=3, random_state=SEED).fit_transform(scaled)
    if kind == "t-SNE + k-means":
        return TSNE(n_components=2, perplexity=20, init="pca", random_state=SEED,
                    max_iter=500).fit_transform(scaled)
    raise ValueError(kind)


def score_pipeline(data, kind):
    """Cluster stability with the entire pipeline refit on every bootstrap replicate.

    cluster_stability resamples the raw fold and calls cluster_fn on each resample,
    so putting embed() inside cluster_fn refits the scaler, PCA or t-SNE from
    scratch on every replicate instead of refitting only k-means. A method with no
    out-of-sample transform is rebuilt per replicate, and its extra instability is
    allowed to show rather than being hidden.
    """
    return cluster_stability(
        data,
        lambda d: KMeans(4, n_init=10, random_state=SEED).fit_predict(embed(d, kind)),
        n_bootstrap=100, random_state=SEED,
    )


METHODS = ["unscaled + k-means", "standardized + k-means", "PCA(2) + k-means",
           "PCA(3) + k-means", "t-SNE + k-means"]

## 2. Cross-validate on held-out folds, shared across methods

Every method sees exactly the same folds, which pairs the comparison and removes between-fold
variance from it, so we generate the grouped K-fold splits once and reuse them. Each pipeline
is fitted and scored inside one held-out fold: the score of a fold depends only on the molecules
in that fold, never on the rows outside it.

We pass `groups=formula`, so `GroupKFold` keeps every constitutional isomer of a formula on one
side of a split; otherwise the folds would share near-identical molecules and every interval below
would be too narrow.

**One partition, and why it is not repeated.** The usual advice is to repeat K-fold
cross-validation over several random partitions. That is impossible here. The 1219 molecules
have only 13 distinct formulae, and the four largest hold 83% of them (group sizes 295, 255, 236
and 222, then 65, 50, 45 and six groups of 12 or fewer). `GroupKFold` assigns groups to folds by
size, so each of the four large formulae anchors a fold and the small ones fill the fifth: test
folds of 295, 255, 236, 222 and 211 molecules, whatever random relabeling of the groups we try. An
earlier version of this notebook ran five "repeats" and got the same five folds each time, so its
25 scores were five values, each counted five times. The cell below builds one partition and
asserts that every test fold is distinct, so identical repeats cannot come back unnoticed.

The five folds are disjoint, so each arm's five scores are analyzed as five independent paired
observations. Five is very little data, and Section 3 shows what that costs.


In [ ]:
from sklearn.model_selection import GroupKFold


def repeated_grouped_splits(n_samples, groups, n_splits=5, n_repeats=1, random_state=0):
    """Grouped K-fold splits, generated once and shared by every arm.

    With n_repeats > 1 each repeat relabels the groups at random. That gives new
    folds only when many groups of similar size can trade places; with the 13
    formulae here every repeat returns the same five folds (Section 2).
    """
    indices = np.arange(n_samples)
    groups = np.asarray(groups)
    for repeat in range(n_repeats):
        # GroupKFold is deterministic, so shuffle which group lands in which
        # fold to try for a different partition on each repeat.
        rng = np.random.default_rng(random_state + repeat)
        remap = {g: i for i, g in enumerate(rng.permutation(np.unique(groups)))}
        shuffled = np.array([remap[g] for g in groups])
        yield from GroupKFold(n_splits=n_splits).split(indices, groups=shuffled)

In [ ]:
splits = list(repeated_grouped_splits(len(X), groups=formula, n_splits=5,
                                      n_repeats=1, random_state=SEED))

# Every test fold must be distinct. With a few large groups a grouped split has
# almost no freedom, and repeats can return the same folds; their scores are then
# copies, and counting copies as observations makes every interval too narrow.
distinct_folds = {frozenset(test.tolist()) for _, test in splits}
assert len(distinct_folds) == len(splits), (
    f"{len(splits)} test folds but only {len(distinct_folds)} distinct: repeated folds "
    "give copies of the same scores, not new observations")
print(f"{len(splits)} held-out folds (one five-fold partition, grouped by molecular formula), "
      f"sizes {sorted((len(test) for _, test in splits), reverse=True)}")

import time
_t0 = time.time()
scores = {m: [] for m in METHODS}
for i, (train, test) in enumerate(splits):
    heldout = X[test]                      # score on the fold the fit never saw, not on training rows
    for method in METHODS:
        scores[method].append(score_pipeline(heldout, method))
    print(f"  fold {i + 1}/{len(splits)} done ({time.time() - _t0:.0f}s elapsed)", end="\r")
print(f"\ndone in {time.time() - _t0:.0f}s")

## 3. Run the protocol

`compare_methods` runs the testing steps of the protocol: a paired test of every arm against a
reference declared in advance, the Holm correction across those tests, effect sizes, and
confidence intervals.

In plain terms: each arm is compared with the reference fold by fold, the test asks whether the
mean per-fold difference is distinguishable from zero, and a multiple-comparison correction then
adjusts the p-values, because testing four arms gives four chances for one of them to come out
significant by luck alone.

There is no omnibus test in front of the pairwise tests. An omnibus test such as Friedman's asks
whether any arm differs from any other, which is not the question declared in advance, and Holm
already controls the error rate across a fixed family of comparisons with one reference, so a
gate in front of it only costs power, which five folds cannot spare.

`print_comparison` writes the result out, and nags about two reporting obligations if you leave
them out: the practical significance threshold and the number of configurations tried.


### What the paired t-test, Holm, Cliff's delta and the intervals each do

**The paired t-test.** Take the five per-fold differences d between an arm and the reference.
The folds of one partition are disjoint, so the test is the ordinary paired t-test:
t = mean(d) / (s / √J), with s the sample standard deviation of d, J = 5 folds and J − 1 = 4
degrees of freedom. `compare_methods` runs it as `dependence="single_partition"`.

The Wilcoxon signed-rank test, the usual distribution-free choice, is no use here: on five pairs
the smallest two-sided p-value it can return is 2/2⁵ = 0.0625, so it cannot reject at 0.05
whatever the data. The t-test assumes the differences are roughly normal, which five values can
neither confirm nor rule out. That is part of the price of having five folds.

**The corrected resampled t-test, for real repeated cross-validation** (Nadeau and Bengio 2003;
Bouckaert and Frank 2004). When K-fold cross-validation is repeated over different partitions,
the repeats reuse the same molecules and the J scores are not independent. A naive paired t-test
estimates the variance of their mean as s²/J; the correction adds a term for the overlap:

    corrected variance of the mean = (1/J + r) · s²,   r = n_test / n_train
    t = mean(d) / sqrt(corrected variance),            J − 1 degrees of freedom

For five repeats of five folds, J = 25 and r = 1/4, and the corrected variance is 7.25 times the
naive one. `compare_methods` runs it as `dependence="repeated_folds"`, and the `corrected_*`
functions below implement it; with r = 0 they reduce to the paired t-test and interval used here.
It is not used on this data because the data cannot be re-partitioned: with 13 formula groups,
four of them holding 83% of the molecules, every repeat of the grouped split returns the same
five folds (Section 2), and copies of the same five scores add no information.

When the scores come from many independent units instead (separate data sets, or independent
draws), `dependence="independent"` runs the Wilcoxon signed-rank test on the paired differences,
which assumes nothing about their distribution. Section 4 uses that mode.

**Holm** controls the family-wise error rate across the comparisons with the reference: the
probability of declaring even one difference that is not there stays below α. It is uniformly more
powerful than Bonferroni and needs no assumption about how the tests depend on one another. With a
short family of comparisons fixed in advance, family-wise control is affordable, and it lets each
"differs from the reference" stand as a claim about that arm.

**Cliff's delta** is the effect size: the probability that a random score from one arm exceeds a
random score from the other, minus the reverse. It assumes nothing about the shape of the
distributions, which suits the small, non-normal per-fold samples you get here. Cohen's d
would assume a normality you have no reason to believe in.

**The intervals** are mean ± t(J − 1, 0.975) × s / √J, with J = 5, both for each arm's mean and
for each arm's difference from the reference. They are per-comparison intervals, not adjusted for
the four comparisons, so an interval for a difference can exclude zero while the Holm-adjusted
test does not reject; the verdict follows Holm. A percentile bootstrap over five scores would be
too narrow, because five values cannot show the tails of their own distribution.


In [ ]:
def corrected_variance(values, test_train_ratio):
    """Variance of the mean of J resampled scores, corrected for overlap between resamples.

    Nadeau and Bengio (2003): (1/J + r) * s^2, with r = n_test / n_train and s^2 the
    sample variance (ddof=1). With r = 0 it is the usual s^2 / J for independent scores.
    """
    x = np.asarray(values, dtype=float)
    return float((1.0 / x.size + test_train_ratio) * x.var(ddof=1))


def corrected_interval(values, test_train_ratio, confidence=0.95):
    """Mean +/- t(J - 1) times the corrected standard error."""
    x = np.asarray(values, dtype=float)
    half = sps.t.ppf(0.5 + confidence / 2, x.size - 1) * np.sqrt(
        corrected_variance(x, test_train_ratio))
    return float(x.mean() - half), float(x.mean() + half)


def corrected_ttest(differences, test_train_ratio):
    """Corrected resampled t-test (Nadeau and Bengio 2003; Bouckaert and Frank 2004).

    Two-sided test of mean(d) = 0 on J paired per-fold differences, with J - 1
    degrees of freedom. Returns (t, p). With test_train_ratio = 0 it is the
    ordinary paired t-test (scipy.stats.ttest_rel).
    """
    d = np.asarray(differences, dtype=float)
    variance = corrected_variance(d, test_train_ratio)
    if variance == 0:                     # identical folds: no spread to test against
        return (0.0, 1.0) if d.mean() == 0 else (float(np.copysign(np.inf, d.mean())), 0.0)
    t = d.mean() / np.sqrt(variance)
    return float(t), float(2 * sps.t.sf(abs(t), d.size - 1))


def cliffs_delta(a, b):
    """Non-parametric effect size in [-1, 1]: P(a > b) minus P(b > a)."""
    x, y = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    greater = np.sum(x[:, None] > y[None, :])
    less = np.sum(x[:, None] < y[None, :])
    return float((greater - less) / (x.size * y.size))


def magnitude(delta):
    """The conventional wording for the size of a Cliff's delta."""
    d = abs(delta)
    if d < 0.147:
        return "negligible"
    if d < 0.33:
        return "small"
    if d < 0.474:
        return "medium"
    return "large"


def holm(p_values, alpha=0.05):
    """Holm step-down family-wise error control. Returns adjusted p-values and rejections."""
    p = np.asarray(p_values, dtype=float)
    m = p.size
    order = np.argsort(p)
    # The i-th smallest p-value is multiplied by (m - i), counting i from zero, and
    # adjusted p-values have to be monotone in the raw ones, so carry the running
    # maximum forward from the smallest.
    adjusted = np.clip(np.maximum.accumulate(p[order] * (m - np.arange(m))), 0, 1)
    out = np.empty(m)
    out[order] = adjusted
    return out, out < alpha


### Follow the protocol step by step

Read `compare_methods` from the top.

Every arm is compared with one reference declared in advance, `PCA(3) + k-means`. That is k − 1
comparisons, and they answer the question set before the data were seen: does any other pipeline
differ from the one we would otherwise use? Testing against whichever arm scored highest would
choose the anchor from the same data the tests then use (Section 9.3).

`dependence` says how the per-fold scores relate to one another, and it picks the test.
`dependence="single_partition"`, used here, is for the disjoint folds of one K-fold partition,
analyzed as independent paired observations: the paired t-test on the differences, with J − 1
degrees of freedom. `dependence="repeated_folds"` is for repeated cross-validation, where the
repeats reuse the same molecules: the corrected resampled t-test, with
`test_train_ratio = n_test / n_train` (0.25 for five-fold). `dependence="independent"` is for scores
from many independent units: the Wilcoxon signed-rank test on the paired differences. All three
tests are paired, comparing each arm with the reference fold by fold, which is the reason for
generating the folds once and sharing them.

Holm then corrects across the k − 1 comparisons, and its rejections alone set `significant`.


In [ ]:
def compare_methods(scores, alpha=0.05, practical_threshold=None,
                    n_configurations_tried=None, reference="PCA(3) + k-means",
                    dependence="repeated_folds", test_train_ratio=0.25):
    """Run the comparison protocol on paired per-fold scores, against a fixed reference.

    Every other arm is tested against `reference`, declared in advance, rather than
    against whichever arm scored highest. Testing against the observed maximum is
    selection on the same data the tests use, the error of Section 9.3; a
    pre-declared reference avoids it. The observed best is still reported, but as a
    description rather than as the arm the tests anchor on.

    `dependence` says how the per-fold scores relate to one another:

    "repeated_folds"    repeated K-fold cross-validation, whose repeats reuse the same
                        molecules. Corrected resampled t-test on the paired
                        differences, with r = test_train_ratio (n_test / n_train,
                        0.25 for five-fold), and intervals from the same variance.
    "single_partition"  the disjoint folds of one K-fold partition, analyzed as
                        independent paired observations. Paired t-test on the
                        differences (J - 1 degrees of freedom), and intervals with r = 0.
    "independent"       scores from many independent units. Wilcoxon signed-rank test
                        on the paired differences, and intervals with r = 0.
    """
    if dependence == "repeated_folds":
        ratio = float(test_train_ratio)
    elif dependence in ("single_partition", "independent"):
        ratio = 0.0
    else:
        raise ValueError("dependence must be 'repeated_folds', 'single_partition' or "
                         f"'independent', got {dependence!r}")

    names = list(scores)
    arrays = {k: np.asarray(v, dtype=float) for k, v in scores.items()}
    if reference not in arrays:
        raise ValueError(f"reference {reference!r} is not one of the arms: {names}")

    lengths = {v.size for v in arrays.values()}
    if len(lengths) != 1:
        raise ValueError(f"a paired design needs equal lengths, got {sorted(lengths)}")
    if not all(np.all(np.isfinite(v)) for v in arrays.values()):
        raise ValueError(
            "non-finite scores: NaN compares False against everything, so a broken "
            "arm can be ranked best. Fix or drop those folds, and say how many you "
            "dropped and why."
        )

    means = {k: float(v.mean()) for k, v in arrays.items()}
    observed_best = max(means, key=means.get)   # descriptive only, not the test anchor
    intervals = {k: corrected_interval(v, ratio) for k, v in arrays.items()}

    # Paired tests, every other arm against the fixed reference, fold by fold.
    comparisons = []
    for name in [k for k in names if k != reference]:
        d = arrays[name] - arrays[reference]           # positive: arm beats the reference
        if dependence in ("repeated_folds", "single_partition"):
            _, p_raw = corrected_ttest(d, ratio)       # r = 0: the ordinary paired t-test
        else:
            p_raw = float(sps.wilcoxon(d).pvalue)
        comparisons.append({
            "method": name,
            "difference": float(d.mean()),
            "difference_ci": corrected_interval(d, ratio),
            "p_raw": p_raw,
            "cliffs_delta": cliffs_delta(arrays[name], arrays[reference]),
        })

    # Holm across the comparisons with the reference, then the effect sizes.
    adjusted, rejected = holm([c["p_raw"] for c in comparisons], alpha=alpha)
    for comp, p_adj, reject in zip(comparisons, adjusted, rejected):
        comp["p_adjusted"] = float(p_adj)
        comp["magnitude"] = magnitude(comp["cliffs_delta"])
        comp["significant"] = bool(reject)

    return {
        "names": names,
        "scores": arrays,
        "intervals": intervals,
        "reference": reference,
        "observed_best": observed_best,
        "dependence": dependence,
        "test_train_ratio": ratio,
        "comparisons": comparisons,
        "alpha": alpha,
        "practical_threshold": practical_threshold,
        "n_configurations_tried": n_configurations_tried,
    }


In [ ]:
def verdict_category(comp, threshold):
    """One of the three verdicts an arm can get against the reference."""
    if not comp["significant"]:
        return "indistinguishable from reference"
    if threshold is not None and abs(comp["difference"]) < threshold:
        return "differs, below practical threshold"
    return "differs, above practical threshold"


def print_comparison(result):
    """Print the comparison, including the parts people leave out."""
    scores, intervals = result["scores"], result["intervals"]
    reference, observed_best = result["reference"], result["observed_best"]
    threshold = result["practical_threshold"]
    if result["dependence"] == "repeated_folds":
        test = f"corrected resampled t-test (r = {result['test_train_ratio']:.2f})"
        interval = "95% CI from the corrected variance"
    elif result["dependence"] == "single_partition":
        test = f"paired t-test ({scores[reference].size - 1} degrees of freedom)"
        interval = "95% CI"
    else:
        test = "Wilcoxon signed-rank on the paired differences"
        interval = "95% CI"

    print(f"Comparison of {len(result['names'])} methods over "
          f"{scores[reference].size} held-out folds (dependence: {result['dependence']})")
    print("=" * 76)

    print(f"\nPer-method scores (mean [{interval}]):")
    for name in result["names"]:
        values = scores[name]
        lo, hi = intervals[name]
        tags = [t for t, on in (("reference", name == reference),
                                ("observed best", name == observed_best)) if on]
        marker = ("  <- " + ", ".join(tags)) if tags else ""
        print(f"  {name:<28s} {values.mean():.4f} [{lo:.4f}, {hi:.4f}]  "
              f"(SD {values.std(ddof=1):.4f}){marker}")
    print(f"\nObserved best is '{observed_best}' (descriptive: it is the highest mean, not the")
    print("arm the tests are anchored on).")

    print(f"\nPaired tests vs the reference '{reference}':")
    print(f"{test}, Holm corrected (positive difference means the arm beats the reference):")
    for comp in result["comparisons"]:
        lo, hi = comp["difference_ci"]
        print(f"  {comp['method']:<26s} diff {comp['difference']:+.4f} [{lo:+.4f}, {hi:+.4f}]  "
              f"p_Holm = {comp['p_adjusted']:.4g}  "
              f"delta = {comp['cliffs_delta']:+.3f} ({comp['magnitude']})")
        print(f"  {'':<26s} -> {verdict_category(comp, threshold)}")

    tied = [c["method"] for c in result["comparisons"] if not c["significant"]]
    if tied:
        print("\n  Indistinguishable from the reference: " + ", ".join(tied))

    print()
    if threshold is None:
        print("Practical significance threshold: NOT DECLARED.")
    else:
        print(f"Practical significance threshold: {threshold:.4f} (derived below)")
    if result["n_configurations_tried"] is None:
        print("Configurations tried: NOT REPORTED.")
    else:
        print(f"Configurations tried: {result['n_configurations_tried']}")


### Derive the practical-significance threshold instead of asserting it

A number pulled from the air, "0.05 stability units," is one nobody can defend, so the threshold
is measured from the metric's own noise. Run two configurations that ought to tie, the
standardized k-means pipeline under two random seeds, on each of five folds, and take the absolute
difference between their stabilities on each fold. The threshold is the 95th percentile of those
five differences. It is a noise floor for the metric: the size of difference the metric produces
from nuisance alone, so a gap smaller than that is not one worth acting on.

The cell also prints the stability the metric returns on a permutation null, where each feature
is shuffled independently so no joint structure remains. That is context for reading the scale,
the value the metric gives when there is nothing to find, and it plays no part in the threshold.


In [ ]:
# A defensible practical-significance threshold, derived rather than guessed.
def permutation_null_features(data, rng):
    """Shuffle each feature independently, destroying joint structure but not the marginals."""
    shuffled = np.array(data, dtype=float)
    for j in range(shuffled.shape[1]):
        rng.shuffle(shuffled[:, j])
    return shuffled


def _standardized_stability(fold, seed):
    return cluster_stability(
        fold,
        lambda d: KMeans(4, n_init=10, random_state=seed).fit_predict(StandardScaler().fit_transform(d)),
        n_bootstrap=100, random_state=seed,
    )

# Context, not part of the threshold: the stability the metric returns on
# structureless data, on each of the five folds.
_rng = np.random.default_rng(SEED)
null_floor = float(np.mean([
    _standardized_stability(permutation_null_features(X[test], _rng), SEED)
    for _, test in splits
]))

# The threshold: the 95th percentile, over five folds, of the absolute stability
# difference between two seeds of the same standardized pipeline. A noise floor.
tie_gaps = [abs(_standardized_stability(X[test], SEED) - _standardized_stability(X[test], SEED + 1))
            for _, test in splits]
tie_spread = float(np.percentile(tie_gaps, 95))

practical_threshold = round(tie_spread, 4)
print(f"permutation-null stability (context)    : {null_floor:.4f}")
print(f"two-seed tie spread (95th percentile |d|): {tie_spread:.4f}")
print(f"derived practical threshold             : {practical_threshold:.4f}")

In [ ]:
report = compare_methods(
    scores,
    practical_threshold=practical_threshold,   # derived just above, not guessed
    n_configurations_tried=len(METHODS),
    reference="PCA(3) + k-means",               # declared in advance; not the observed best
    dependence="single_partition",              # one five-fold partition: disjoint folds
)
print_comparison(report)

In [ ]:
# For comparison, the stability t-SNE + k-means appears to have when the embedding is
# fit once: fit t-SNE on each standardized fold, freeze that embedding, and bootstrap
# only k-means on it. That measures the stability of one picture rather than of the
# procedure, which is why the comparison
# above refits the whole pipeline on every replicate.
frozen_tsne = [
    cluster_stability(
        embed(X[test], "t-SNE + k-means"),
        lambda d: KMeans(4, n_init=10, random_state=SEED).fit_predict(d),
        n_bootstrap=100, random_state=SEED,
    )
    for _, test in splits
]
print(f"t-SNE + k-means, embedding fit once, k-means resampled : {np.mean(frozen_tsne):.3f}")
print(f"t-SNE + k-means, whole pipeline refit per replicate     : {np.mean(scores['t-SNE + k-means']):.3f}")


In [ ]:
# --- Numbers the article quotes ------------------------------------------------
# The whole pipeline is refit on every bootstrap replicate, on one grouped five-fold
# partition: each arm against PCA(3) + k-means, paired t-test on the five per-fold
# differences, Holm.
means = {m: float(np.mean(v)) for m, v in scores.items()}
ranking = sorted(means, key=means.get, reverse=True)
verdicts = {c["method"]: verdict_category(c, practical_threshold) for c in report["comparisons"]}
p_holm = {c["method"]: c["p_adjusted"] for c in report["comparisons"]}
print("mean held-out-fold stability, 95% CI, Holm p vs PCA(3) + k-means, verdict:")
for m in ranking:
    lo, hi = report["intervals"][m]
    tail = (f"p_Holm = {p_holm[m]:.3g}  {verdicts[m]}" if m in verdicts else "reference")
    print(f"  {m:26s} {means[m]:.4f} [{lo:.3f}, {hi:.3f}]  {tail}")
print(f"\nderived practical threshold: {practical_threshold:.4f}   article: 0.042")

# Standardized against unscaled is not one of the pre-declared comparisons, so it is
# reported descriptively, with the same paired t-test and interval.
d_su = np.asarray(scores["standardized + k-means"]) - np.asarray(scores["unscaled + k-means"])
lo, hi = corrected_interval(d_su, 0.0)
print(f"standardized - unscaled (descriptive): {d_su.mean():+.4f} [{lo:+.3f}, {hi:+.3f}], "
      f"paired t-test p = {corrected_ttest(d_su, 0.0)[1]:.2f}")

assert means["t-SNE + k-means"] == min(means.values()), "t-SNE is no longer the least stable arm"
assert means["t-SNE + k-means"] < 0.55, f"t-SNE stability drifted: {means['t-SNE + k-means']:.3f}"
assert ranking[0] in ("unscaled + k-means", "PCA(3) + k-means"), f"top arm changed: {ranking[0]!r}"
assert means["standardized + k-means"] < means["unscaled + k-means"], (
    "standardizing no longer gives the lower mean stability")
# The frozen-embedding t-SNE score the article quotes as 0.83. It is still above
# every refit arm, which is the article's point.
frozen_mean = float(np.mean(frozen_tsne))
print(f"t-SNE fit once, k-means resampled: {frozen_mean:.3f}   article: 0.83 "
      f"(refit: {means['t-SNE + k-means']:.3f})")
assert abs(frozen_mean - 0.83) <= 0.08, f"frozen t-SNE stability drifted: {frozen_mean:.3f}"
assert frozen_mean > max(means.values()), (
    "fitting t-SNE once no longer scores above every refit arm")
assert abs(practical_threshold - 0.042) < 0.005, f"practical threshold drifted: {practical_threshold:.4f}"
assert verdicts["unscaled + k-means"] == "indistinguishable from reference", (
    f"unscaled + k-means is now '{verdicts['unscaled + k-means']}'; the article says it is not distinguishable")
assert verdicts["PCA(2) + k-means"] == "indistinguishable from reference", (
    f"PCA(2) + k-means is now '{verdicts['PCA(2) + k-means']}'")
assert verdicts["standardized + k-means"] == "indistinguishable from reference", (
    f"standardized + k-means is now '{verdicts['standardized + k-means']}'; with five folds "
    "it was not distinguishable from the reference")
assert verdicts["t-SNE + k-means"] == "differs, above practical threshold" and p_holm["t-SNE + k-means"] < 0.05, (
    "t-SNE is no longer significantly and practically less stable than the reference")
print("within tolerance")


### Read the report carefully

Five things in the output above repay a second look.

**t-SNE is the least stable arm, and the only one that separates from the reference.** When the
whole pipeline is refit on every bootstrap replicate, t-SNE, which has no out-of-sample
transform, is rebuilt from scratch on each resample and lands in a different arrangement every
time, so k-means on it agrees with itself poorly across resamples (stability about 0.47). It sits
about 0.28 below the reference on average, below it in every fold (by 0.17 to 0.40), its
interval overlaps no other arm's, and it differs from the reference after Holm (p about 0.01)
and by far more than the practical threshold. Fit the embedding once and resample only k-means,
as the cell after the report does, and t-SNE scores about 0.83, above every other arm; that
measures the stability of a frozen picture rather than of the procedure.

**Five folds give little power, and every other arm is indistinguishable from the reference.**
That verdict says as much about the design as about the pipelines. Five folds are five paired
observations; a t-test with four degrees of freedom needs a t of 2.78 to reject at 0.05, and Holm
then asks more of all but the strongest comparison. Indistinguishable means the data cannot tell
these arms apart, not that they are equivalent: the difference intervals for unscaled and PCA(2),
about −0.18 to +0.25 and −0.16 to +0.09, hold differences several times the practical threshold.
More folds cannot be had by repeating the split, because the 13 formula groups fix it
(Section 2); they would have to come from more molecules, or more formulae.

**Standardized k-means is the closest call, and it does not separate either.** Its difference
from the reference, −0.046, is negative in all five folds, and its 95% interval,
−0.089 to −0.004, excludes zero; its unadjusted p-value is 0.04. Holm multiplies that by three, as
the second-smallest of four, to 0.12. The interval is a per-comparison statement and Holm a
family-wise one, so the two can disagree, and the verdict follows Holm. The gap is also only just
over the practical threshold of 0.042. Against unscaled, a comparison that was not declared in
advance and is printed only descriptively in the cell above, its mean is lower by about 0.08,
with an interval from −0.31 to +0.15. "Standardizing lowers stability" is a statement about the
means here, not one the data supports as a difference.

**The unscaled pipeline has the highest mean, and the test cannot separate it from the
reference.** Its per-fold scores are the most variable of the five (SD about 0.12), and
its difference from the reference, about +0.03, has an interval reaching from well below zero to
well above it. Even where it scores high, that is not a good thing in itself: a handful of
enormous features (the close contacts of the heaviest atom present) impose a partition that is
trivially reproducible and chemically almost vacuous, the artifact Figure 3 of the article shows
PCA recovering. Stability is necessary evidence, not sufficient evidence.

**We test against a pre-declared reference, and the threshold is a noise floor.** Every arm is
compared to `PCA(3) + k-means`, chosen in advance, because testing against whichever arm scored
highest would be selection on the same data the test uses (Section 9.3). The practical threshold
is the 95th percentile, over the five folds, of the absolute stability difference between two
seeds of the same standardized pipeline: the size of difference the metric produces from nuisance
alone. Every arm uses the same k-means at the same k, so the spread from best to worst comes
entirely from what was done to the features first.


### Draw the comparison so the verdict is visible

Show the distributions, not a table of bolded means. The plot puts every fold's cluster-stability
value on the axis, higher meaning the clusters recur more reliably, with the mean and its 95%
confidence interval on top of them, and colors each arm by its verdict against
the reference.

Four categories stay apart, because collapsing them is the reporting failure this figure exists
to prevent: the reference; the arms the data cannot separate from it; the arms that differ
significantly but by less than the practical threshold you declared; and the arms that differ
both significantly and by more than the threshold. Only that last group gives you grounds to
prefer one method over another.

Color carries the verdict and the dash pattern repeats it, so the figure still reads in a
black-and-white print.


In [ ]:
def plot_comparison(result, ax=None, title="Method comparison"):
    """Per-fold scores with means, 95% intervals, and the verdict vs the reference."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 3.2))

    reference = result["reference"]
    order = sorted(result["names"], key=lambda n: result["scores"][n].mean(), reverse=True)
    threshold = result["practical_threshold"]
    verdicts = {c["method"]: verdict_category(c, threshold) for c in result["comparisons"]}

    for i, name in enumerate(order):
        values = result["scores"][name]
        if name == reference:
            color, dashes, marker, label = GREEN, "-", "s", "reference"
        elif verdicts[name] == "indistinguishable from reference":
            color, dashes, marker, label = (TEAL, (0, (4, 1.5)), "o",
                                            "indistinguishable from reference")
        elif verdicts[name] == "differs, below practical threshold":
            color, dashes, marker, label = (PURPLE, (0, (1, 1.2)), "v",
                                            "differs, below practical threshold")
        else:
            color, dashes, marker, label = (PLUM, (0, (5, 1.2, 1, 1.2)), "^",
                                            "differs, above practical threshold")

        jitter = np.random.default_rng(i).uniform(-0.12, 0.12, size=len(values))
        ax.scatter(values, np.full(len(values), i) + jitter, s=9, color=color,
                   alpha=0.35, linewidths=0, zorder=2, marker=".")
        lo, hi = result["intervals"][name]
        # One simple marker per verdict category (circle / square / triangle),
        # set explicitly so it tracks the category and not the row-order marker
        # cycle. markevery=[] keeps markers off the interval line while leaving
        # them on the legend handle; the mean marker below carries the shape.
        ax.plot([lo, hi], [i, i], color=color, lw=2.5, linestyle=dashes, zorder=3,
                solid_capstyle="butt", marker=marker, markevery=[],
                label=label if label not in ax.get_legend_handles_labels()[1] else None)
        ax.scatter([values.mean()], [i], s=48, color=color, zorder=4,
                   marker=marker, edgecolor="white", linewidth=1)

    # Extend the x-axis past the extreme arms so no points sit on the frame:
    # the least stable arm and the high tail of the highest arm both need margin.
    all_vals = np.concatenate([result["scores"][n] for n in result["names"]])
    span = all_vals.max() - all_vals.min()
    ax.set_xlim(all_vals.min() - 0.06 * span, all_vals.max() + 0.06 * span)

    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order)
    ax.invert_yaxis()
    ax.set_xlabel("cluster stability per fold (label-free reproducibility, not accuracy)")
    ax.legend(loc="lower right", fontsize=8)
    if title:
        ax.set_title(title)
    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
plot_comparison(report, ax=ax)
plt.show()

## 4. See what a naive comparison does without the protocol

Here is the same comparison done badly: unpaired, uncorrected, and judged on means. We run five
*identical* pipelines under different names, the same number of arms as the comparison above, and
count how often a naive best-versus-worst test declares one of them the winner. On data with no
real differences a valid procedure should do so at most 5% of the time.

These arms are independent draws, not repeated folds of one data set, so the protocol runs with
`dependence="independent"`: the Wilcoxon signed-rank test against `arm 0`, Holm corrected.

In [ ]:
# Five arms, the same number the comparison above used, drawn from ONE
# distribution so there is no real difference to find. A thousand trials pins
# the rate down: on 60 trials the two procedures are within sampling noise of
# each other, which is exactly the mistake this section is about.
n_arms = 5
trials = 1000

naive_wins = 0
protocol_wins = 0
for seed in range(trials):
    local = np.random.default_rng(2000 + seed)
    arms = {f"arm {i}": local.normal(0.62, 0.04, 15) for i in range(n_arms)}

    # Naive: pick the best mean, run an uncorrected t-test against the worst.
    best = max(arms, key=lambda k: arms[k].mean())
    worst = min(arms, key=lambda k: arms[k].mean())
    if sps.ttest_ind(arms[best], arms[worst]).pvalue < 0.05:
        naive_wins += 1

    # Protocol: every arm against a reference fixed in advance, Holm corrected. The
    # arms are independent draws, not repeated folds, so there is no overlap to correct.
    rep = compare_methods(arms, reference="arm 0", dependence="independent")
    if any(c["significant"] for c in rep["comparisons"]):
        protocol_wins += 1

print(f"Over {trials} trials on {n_arms} identical arms (NO real differences):")
print(f"  naive best-vs-worst t-test declared a winner : {100 * naive_wins / trials:5.1f}% of the time")
print(f"  the full protocol declared a winner          : {100 * protocol_wins / trials:5.1f}% of the time")
print()
print("The naive rate is inflated because 'compare the best against the worst'")
print("is a selection over five arms, and the t-test does not know a selection")
print("happened. The protocol holds near the nominal 5%.")

assert protocol_wins / trials <= 0.065, (
    f"the protocol declared a winner in {protocol_wins / trials:.1%} of null trials; "
    "Holm against a fixed reference should hold at or below about 5%")

---
## 5. Which test for which question?

The protocol above answers "is method A better than B". Chemists ask other questions too, and
the appropriate procedure differs (manuscript Table 5). Two of them are done here, because the
obvious approach is wrong in both cases.

### 5.1 Is an observed ARI significantly better than chance?

The adjusted Rand index is corrected for chance *in expectation*, so a value near 0 means "no
better than random". But ARI has no sampling distribution attached, so an observed 0.15 cannot be
called significant by inspection. A permutation test supplies the missing piece.

The function below scores two label vectors with ARI or AMI, then shuffles one of them a few
hundred times to build the null the index does not come with. The p-value counts the observed
value in among the permutations, which is why the smallest value it can report is
1 / (n_permutations + 1). Ask for 999 permutations and you can resolve down to 0.001; ask for 99
and you cannot.

In [ ]:
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score


def partition_agreement(labels_a, labels_b, index="ari", n_permutations=999,
                        random_state=0):
    """Agreement between two partitions of the same objects, with a p-value."""
    score = {"ari": adjusted_rand_score, "ami": adjusted_mutual_info_score}[index]
    a, b = np.asarray(labels_a), np.asarray(labels_b)
    observed = float(score(a, b))

    rng = np.random.default_rng(random_state)
    permuted = b.copy()
    null = np.empty(n_permutations)
    for i in range(n_permutations):
        rng.shuffle(permuted)
        null[i] = score(a, permuted)

    return {"index": index, "observed": observed, "null_mean": float(null.mean()),
            "p_value": (int(np.sum(null >= observed)) + 1) / (n_permutations + 1)}


def agreement_line(res):
    """One line describing an agreement test."""
    verdict = ("agreement exceeds chance" if res["p_value"] < 0.05
               else "NO better than chance")
    return (f"{res['index'].upper()} = {res['observed']:+.3f} "
            f"(null {res['null_mean']:+.3f}, p = {res['p_value']:.3f}) -> {verdict}")

In [ ]:
from sklearn.cluster import AgglomerativeClustering

# Two clusterings of the same molecules, from different algorithms.
Xs = StandardScaler().fit_transform(X)
lab_km = KMeans(4, n_init=10, random_state=SEED).fit_predict(Xs)
lab_ward = AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(Xs)
lab_random = rng.permutation(lab_km)

for name, other in [("Ward", lab_ward), ("random relabeling", lab_random)]:
    res = partition_agreement(lab_km, other, n_permutations=999, random_state=SEED)
    print(f"k-means vs {name:20s} {agreement_line(res)}")
print()
print("Look at the second line: a random relabeling gives ARI near zero AND a large p-value.")
print("That is the calibration check: the test must fail to reject when nothing is there.")

**Which index?** ARI when the reference partition has large, roughly equal clusters; AMI when it
is unbalanced with small clusters (Romano et al. 2016). If a conclusion depends on the choice,
report both, and be aware that the chance-correction assumes a fixed-margins null which is
itself a modeling assumption (Gates & Ahn 2017).

In [ ]:
print("Same comparison under both indices:")
for index in ("ari", "ami"):
    res = partition_agreement(lab_km, lab_ward, index=index, n_permutations=499,
                              random_state=SEED)
    print(f"  {agreement_line(res)}")

### 5.2 Do my clusters differ in a measured property? (The one everybody gets wrong)

This is the analysis chemists actually run: cluster the molecules, attach an activity, run a
one-way ANOVA, report that the clusters differ significantly.

A significant result shows that the property varies across feature space. It cannot show that the
clusters exist. To see the difference, we run the analysis on data with *no cluster structure
whatsoever*, under three relationships between the property and the features you clustered on.


In [ ]:
def anova_after_clustering(mode, n=300, d=10, k=3, seed=0, rho=0.5):
    """Cluster structureless data, then ANOVA a property across the clusters.

    mode = 'used'       : the property IS one of the clustered features
    mode = 'correlated' : the property correlates with a clustered feature
    mode = 'external'   : the property is independent of everything
    """
    r = np.random.default_rng(seed)
    X = r.standard_normal((n, d))            # NO cluster structure, by construction
    if mode == "used":
        prop = X[:, 0]
    elif mode == "correlated":
        prop = rho * X[:, 0] + np.sqrt(1 - rho**2) * r.standard_normal(n)
    else:
        prop = r.standard_normal(n)
    labels = KMeans(k, n_init=10, random_state=seed).fit_predict(X)
    return sps.f_oneway(*[prop[labels == c] for c in range(k)]).pvalue


print("Rejection rate of a naive ANOVA across k-means clusters, on data with NO clusters:\n")
rates = {}
for mode in ("used", "correlated", "external"):
    p = np.array([anova_after_clustering(mode, seed=s) for s in range(200)])
    rates[mode] = (p < 0.05).mean()
    print(f"  property {mode:12s} {rates[mode]:6.1%}   (median p = {np.median(p):.2e})")

Read that table carefully, because it contains the whole lesson.

- When the property **is one of the features you clustered on**, the test rejects almost always.
  Clusters *are* regions of that feature space, so of course they differ in it. The rejection is
  correct, and it says nothing about whether the clusters exist.
- When the property is **independent of everything**, the test rejects at about the nominal 5%.
  Here, and only here, the rejection rate is a false-positive rate in the ordinary sense.
- The realistic case is in between: a property that is correlated with the features.


In [ ]:
print("Correlation between the property and a clustered feature vs. rejection rate:\n")
for rho in (0.0, 0.1, 0.2, 0.3, 0.5):
    p = np.array([anova_after_clustering("correlated", seed=s, rho=rho) for s in range(200)])
    bar = "#" * int(60 * (p < 0.05).mean())
    print(f"  rho = {rho:.1f}   {(p < 0.05).mean():6.1%}  {bar}")

**What a rejection does and does not show.** In the correlated case the ANOVA is not wrong. The
property really does vary across feature space, the clusters are regions of feature space, and so
the regions differ in the property. The test rejects because there is an association to detect,
and the stronger the association, the more often it detects it. What the rejection does not
license is the conclusion usually drawn from it, that distinct groups exist and differ in the
property. These data have no groups at all.

So the quantity above is a **rejection rate**. It is a false-positive rate only for the claim that
clusters exist, and that is a claim the ANOVA does not test.

### 5.3 What each analysis can establish

**Hold out data the clusters did not see, when the property influenced any choice.** If the
property guided the number of clusters, the metric, the method, or which configuration was kept,
the in-sample test reuses that information. Then cluster on a training split, assign the
held-out molecules to the nearest centroid, and test the property there: the regions are fixed
before the held-out molecules are seen, so the test asks a valid, narrower question, *do these
fixed regions of feature space differ in the property?* When the property influenced none of
those choices, the in-sample ANOVA is already a valid test of association between the property
and the features, and splitting mainly costs power. Either way, each molecule is assigned by its
own features, so a property correlated with those features differs between the groups when the
data has no clusters. Those rejections are correct detections of the association, and they grow
with sample size. The cell below measures both tests at n = 300 and at n = 1200.

**Selective inference** (Gao et al. 2024 for hierarchical clustering, Chen & Witten 2023 for
k-means, Chen & Gao 2025 for a single feature) concerns features that were used in the
clustering: it tests a difference in such a feature between two clusters, conditioning on the
event that the algorithm produced those clusters. Reference implementations exist in R; we are
not aware of a Python equivalent.

**Report descriptively** when neither applies: state the effect size and say that the p-value is
uncorrected for selection.

Neither test answers "do distinct groups differ in this property?" on its own. That claim needs
two separate pieces of evidence: that the groups exist, from the features alone (a null model and
stability, notebook 08), and that the regions differ in the property, from a test that is valid
for how the clusters were chosen, reported with an effect size.


In [ ]:
def heldout_after_clustering(n=300, d=10, k=3, seed=0, rho=0.3):
    r = np.random.default_rng(seed)
    X = r.standard_normal((n, d))
    prop = rho * X[:, 0] + np.sqrt(1 - rho**2) * r.standard_normal(n)
    idx = r.permutation(n)
    train, test = idx[: n // 2], idx[n // 2 :]
    km = KMeans(k, n_init=10, random_state=seed).fit(X[train])
    labels = km.predict(X[test])          # partition fixed before these points were seen
    groups = [prop[test][labels == c] for c in range(k)]
    groups = [g for g in groups if len(g) > 1]
    return sps.f_oneway(*groups).pvalue if len(groups) > 1 else 1.0


# Both tests, four strengths of association, two sample sizes, 200 data sets each.
# The data never has clusters; at rho > 0 the property does vary across feature space.
RHOS, SIZES = (0.0, 0.2, 0.3, 0.5), (300, 1200)
rejection, pvalues = {}, {}
for n in SIZES:
    for rho in RHOS:
        pvalues[("naive", n, rho)] = np.array(
            [anova_after_clustering("correlated", n=n, seed=s, rho=rho) for s in range(200)])
        pvalues[("held-out", n, rho)] = np.array(
            [heldout_after_clustering(n=n, seed=s, rho=rho) for s in range(200)])
rejection = {key: float((p < 0.05).mean()) for key, p in pvalues.items()}

print("Rejection rate at p < 0.05, property correlated with a clustered feature, no clusters:\n")
print(f"  {'rho':>5}" + "".join(f"{f'{t}, n = {n}':>20}" for n in SIZES for t in ("naive", "held-out")))
for rho in RHOS:
    print(f"  {rho:5.1f}" + "".join(f"{rejection[(t, n, rho)]:20.1%}"
                                    for n in SIZES for t in ("naive", "held-out")))

naive, held = pvalues[("naive", 300, 0.3)], pvalues[("held-out", 300, 0.3)]
fig, ax = plt.subplots(figsize=(5.5, 3))
bins = np.linspace(0, 1, 21)
ax.hist(naive, bins=bins, alpha=0.65, density=True, label=f"naive ({(naive < 0.05).mean():.0%} rejected)")
ax.hist(held, bins=bins, alpha=0.65, density=True, label=f"held-out ({(held < 0.05).mean():.0%} rejected)")
ax.axhline(1.0, color="gray", ls="--", lw=1)
ax.text(0.5, 1.08, "uniform = no association to detect", fontsize=7, color="gray")
ax.set_xlabel("p-value"); ax.set_ylabel("density")
ax.set_title("Property correlated with the features (rho = 0.3), data with no clusters")
ax.legend(frameon=False, fontsize=8)
plt.show()
print("Neither histogram is uniform, and neither should be: the property is correlated with the")
print("features, so the regions of feature space really do differ in it, and both tests detect")
print("that more often as n grows. At rho = 0 the held-out test sits near 5%. A rejection here")
print("is evidence that the property varies across feature space, not that clusters exist, so")
print("the claim 'distinct groups differ in this property' needs two separate pieces of evidence:")
print("  1. that groups exist, from the features alone (null model and stability, notebook 08)")
print("  2. that the regions differ in the property, from a held-out test, with an effect size")


In [ ]:
# --- Numbers the article quotes -------------------------------------------
# These assertions exist because continuous integration proves the notebook
# *runs*; it does not prove it still says what the article says it says. A
# library default changes, a result shifts, CI stays green, and the article is
# quietly wrong. Tolerance bands, not equality: catch a change that matters,
# not floating-point noise. See Section 11.3 on dependency rot.
#
# Rejection rates are over 200 data sets, so their binomial standard error is
# up to 3.5 percentage points; the bands below are about twice that.

print(f"naive ANOVA, property used         {rates['used']:8.1%}   article: >90%")
print(f"naive ANOVA, property external     {rates['external']:8.1%}   article: ~5% (calibrated)")

ARTICLE = {                                   # (test, n, rho): rejection rate the article quotes
    ("naive", 300, 0.2): 0.20, ("naive", 300, 0.5): 0.64,
    ("held-out", 300, 0.2): 0.11, ("held-out", 300, 0.5): 0.46,
    ("naive", 1200, 0.2): 0.56, ("naive", 1200, 0.5): 0.89,
}
for (test, n, rho), quoted_rate in ARTICLE.items():
    got = rejection[(test, n, rho)]
    print(f"{test:>8s} ANOVA, n = {n:4d}, rho = {rho:.1f}   {got:8.1%}   article: {quoted_rate:.0%}")
    assert abs(got - quoted_rate) <= 0.07, (
        f"{test} rejection rate at n = {n}, rho = {rho} drifted to {got:.1%}; "
        f"the article quotes {quoted_rate:.0%}")

assert rates["used"] >= 0.87, f"circular ANOVA rejection rate drifted: {rates['used']:.1%}; the article says over 90%"
assert rates["external"] < 0.15, f"independent property no longer calibrated: {rates['external']:.1%}"
assert rejection[("held-out", 300, 0.0)] < 0.10 and rejection[("held-out", 1200, 0.0)] < 0.10, (
    "the held-out test no longer holds its level when there is no association")
for rho in (0.2, 0.3, 0.5):
    assert rejection[("held-out", 300, rho)] < rejection[("naive", 300, rho)], (
        f"holding out no longer lowers the rejection rate at rho = {rho}")
    for test in ("naive", "held-out"):
        assert rejection[(test, 1200, rho)] > rejection[(test, 300, rho)], (
            f"the {test} rejection rate no longer grows with n at rho = {rho}; "
            "the article's point that these are detections of a real association depends on it")
print("\nall within tolerance of the values printed in the article")


## Manuscript figure


- **This is Figure 9 of the manuscript** (`method_comparison`): five clustering pipelines on QM7 compared by the full statistical protocol, with each arm's verdict against the pre-declared reference, PCA(3) + k-means.


This cell produces `method_comparison`, the article's Section 9 figure: the comparison computed in
sections 2 and 3 above, drawn as the five per-fold points of each arm with its mean and 95%
confidence interval, colored by each arm's verdict against the pre-declared reference. It uses the `report` object from
section 3 instead of recomputing it, so the figure in the article and the numbers in this notebook
cannot disagree. It is written as **both a PDF and a PNG**.

The published version carries no plot title, because that text belongs in the caption, so the cell
passes `title=None`. The last line calls `set_style()` again to put the screen defaults back, so
the plot in section 3 still behaves as it did if you re-run it.

In [ ]:
# --- Manuscript figure: method_comparison (Section 9).
#
# The figure code lives here rather than in a script, so that the notebook a
# reader follows and the figure the article prints cannot drift apart.

import os


def savefig(fig, stem, outdir=None):
    """Write a figure as both a PNG and a PDF to a local ``figures/`` directory.

    Set the FIGDIR environment variable, or pass outdir, to write somewhere else.
    """
    outdir = Path(outdir or os.environ.get("FIGDIR") or "figures")
    outdir.mkdir(parents=True, exist_ok=True)
    for fmt in ("png", "pdf"):
        fig.savefig(outdir / f"{stem}.{fmt}")
    return outdir


# scripts/make_manuscript_figures.py sets FIGDIR to collect the figures; without
# it, savefig falls back to the local figures/ directory.
FIGDIR = os.environ.get("FIGDIR")

# Figures are sized for a two-column LaTeX layout, so the type has to be
# smaller than a screen-oriented default. 7.5 pt on this canvas is 7.5 pt on
# the page, because the figure is authored at exactly the width LaTeX includes
# it at and so is never rescaled.
mpl.rcParams.update(
    {
        "figure.dpi": 200,
        "font.size": 7.5,
        "axes.labelsize": 7.5,
        "axes.titlesize": 8,
        "legend.fontsize": 7,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.7,
        "ytick.major.width": 0.7,
        "xtick.major.size": 2.8,
        "ytick.major.size": 2.8,
        "lines.linewidth": 1.3,
    }
)

# Canonical figure width, in inches, measured from the compiled document:
# \linewidth = 250.95 pt = 3.47 in, one column of the two-column layout.
COL_W = 3.47

# The report is the one computed in section 3, on the 1219 fifteen-atom QM7
# molecules: five pipelines, five paired held-out folds (one five-fold partition)
# grouped by molecular formula, scored by label-free cluster stability. It is reused, not
# recomputed, so the article and the notebook cannot drift apart.
fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.9))
plot_comparison(report, ax=ax, title=None)
ax.set_xlabel("cluster stability per fold\n(mean and 95% CI)", fontsize=7)
ax.tick_params(axis="y", labelsize=6.6)
# Legend inside the axes, lower right, where the low-stability arms leave the
# corner empty. A near-opaque background keeps it readable over any stray point.
ax.legend(
    loc="lower right", fontsize=6.0, handlelength=1.8, labelspacing=0.35,
    framealpha=0.92, edgecolor="none", borderaxespad=0.5,
)
outdir = savefig(fig, "method_comparison", FIGDIR)
print(f"wrote method_comparison.png/.pdf to {outdir}")
plt.show()

# Leave the notebook as we found it, so anything you add below keeps its title
# and its screen-sized type.
set_style()

### Exercise

Build the splits with `n_repeats=5`, comment out the distinct-fold assertion, and count the
distinct test folds. Pass the 25 scores to `compare_methods`, once with
`dependence="independent"` and once with `dependence="repeated_folds"`, and compare the intervals
and verdicts with the ones above. Which treats copies of the same five scores as new evidence, and
does the corrected variance undo that? Then work out the smallest two-sided p-value a Wilcoxon
signed-rank test can return on the five pairs of the real design. What does it tell you about
papers that compare methods on a single five-fold split with a rank test?


In [ ]:
# YOUR CODE HERE